# 阶段三：结构化并发

使用 `asyncio.TaskGroup` 管理一批工具任务。批次结束时，所有子任务都必须结束，不留下失控的后台任务。

In [ ]:
import asyncio


async def fake_tool(name: str, delay: float, should_fail: bool = False) -> str:
    """模拟一个异步工具。

    参数:
        name: 工具名称。
        delay: 工具运行前等待的秒数。
        should_fail: 是否在等待后抛出异常。

    返回:
        工具成功时返回结果字符串。

    异常:
        RuntimeError: `should_fail` 为 True 时抛出。
    """
    try:
        await asyncio.sleep(delay)
        if should_fail:
            raise RuntimeError(f"{name} failed")
        return f"{name}: done"
    except asyncio.CancelledError:
        print(f"{name}: cancelled")
        raise


async def run_batch(task_refs: list[asyncio.Task[None]]) -> list[str]:
    """在一个结构化并发作用域中运行工具批次。

    参数:
        task_refs: 用于记录本批次创建的子任务，便于验收任务生命周期。

    返回:
        所有工具成功时的结果列表。

    异常:
        ExceptionGroup: 任一工具失败时，`TaskGroup` 取消同批任务并传播异常组。
    """
    results: list[str] = []

    async def collect(name: str, delay: float, should_fail: bool = False) -> None:
        result = await fake_tool(name, delay, should_fail)
        results.append(result)

    async with asyncio.TaskGroup() as group:
        task_refs.extend([
            group.create_task(collect("search", 0.1)),
            group.create_task(collect("weather", 0.3)),
            group.create_task(collect("database", 0.2, should_fail=True)),
        ])

    return results


async def main() -> None:
    """运行验收：失败任务会取消仍在运行的同批任务。"""
    task_refs: list[asyncio.Task[None]] = []
    try:
        await run_batch(task_refs)
    except ExceptionGroup as exc_group:
        print(f"batch failed with {len(exc_group.exceptions)} exception(s)")

    assert all(task.done() for task in task_refs)
    print("batch finished without background tasks")


await main()


batch finished without background tasks


In [ ]:
from langchain_core.runnables import RunnableLambda

async def langchain_async_example() -> None:
    """把 LangChain Runnable 放入 TaskGroup，统一管理子任务。"""
    async def run(name: str) -> str:
        return await fake_tool(name, 0.01)

    runnable = RunnableLambda(afunc=run)
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(runnable.ainvoke(name)) for name in ["search", "weather"]]
    results = [task.result() for task in tasks]
    print("LangChain Async:", results)
    assert results == ["search: done", "weather: done"]

await langchain_async_example()


In [3]:
from fastapi import FastAPI
app = FastAPI()

@app.post("/phase3/tools")
async def run_phase3_tools() -> list[str]:
    """提供 HTTP 入口，调用结构化并发批次。"""
    task_refs: list[asyncio.Task[None]] = []
    return await run_batch(task_refs)

assert any(route.path == "/phase3/tools" for route in app.routes)
print("FastAPI route registered")


FastAPI route registered
